# Supplementary Table 1 - Runtime and memory benchmarking

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** A100 `/data/taobo.hu/projects/mouse_pup/benchmarking_mouse_pup.ipynb`, `tmp_coste_perf.py`, and `tmp_squidpy_ne.py`.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Supplementary Table 1, shared setup. Imports libraries used for timing, memory profiling, and benchmark result tables.
#

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
# Panel mapping:
# - Supplementary Table 1, Squidpy benchmark helper. Builds AnnData and computes Squidpy NE so runtime and memory are measured for the full comparator method.
#

import anndata as ad
import squidpy as sq


def dataframe_to_adata(df):
    """Convert a point table into a minimal AnnData object for neighborhood methods."""
    adata = ad.AnnData(X=np.zeros((df.shape[0], 1), dtype=float))
    adata.obs["celltype"] = pd.Categorical(df["celltype"].to_numpy())
    adata.obsm["spatial"] = df[["x", "y"]].to_numpy()
    adata.obs_names = [f"cell_{i}" for i in range(df.shape[0])]
    return adata


def compute_squidpy_ne(df, n_perms=1000, radius=None, n_neighs=None):
    """Run Squidpy neighborhood enrichment and return its z-score matrix."""
    adata = dataframe_to_adata(df)
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial", radius=radius, n_neighs=n_neighs)
    sq.gr.nhood_enrichment(adata, cluster_key="celltype", n_perms=n_perms)
    cats = adata.obs["celltype"].cat.categories
    z = adata.uns["celltype_nhood_enrichment"]["zscore"]
    return pd.DataFrame(z, index=cats, columns=cats)


In [ ]:
# Panel mapping:
# - Supplementary Table 1, Giotto benchmark helper. Wraps Giotto::cellProximityEnrichment so Giotto timing and memory can be recorded.
#

def compute_giotto_matrix(df, n_simulations=1000, k=6):
    """Run Giotto cell proximity enrichment and return a square score matrix.

    The manuscript benchmark used Giotto through the R package interface. This
    wrapper keeps the same data contract as the Python benchmark helpers:
    an input table with x, y, and celltype columns is converted to a minimal
    Giotto object, a kNN spatial network is created, and
    Giotto::cellProximityEnrichment is run on the celltype labels.
    """
    try:
        import rpy2.robjects as ro
        from rpy2.robjects import pandas2ri
        from rpy2.robjects.conversion import localconverter
    except ImportError as exc:
        raise ImportError("Install rpy2 and the R Giotto package to run this cell.") from exc

    r_code = """
    function(point_table, k, n_simulations) {
      suppressPackageStartupMessages(library(Giotto))
      point_table <- as.data.frame(point_table)
      point_table$cell_ID <- as.character(seq_len(nrow(point_table)))
      spatial_locs <- point_table[, c("cell_ID", "x", "y")]
      expr <- matrix(0, nrow = 1, ncol = nrow(point_table))
      rownames(expr) <- "dummy"
      colnames(expr) <- point_table$cell_ID
      g <- createGiottoObject(raw_exprs = expr, spatial_locs = spatial_locs)
      pDataDT(g)$celltype <- as.character(point_table$celltype)
      g <- createSpatialNetwork(gobject = g, method = "kNN", k = as.integer(k), name = "spatial_network")
      enrich <- cellProximityEnrichment(
        gobject = g,
        cluster_column = "celltype",
        spatial_network_name = "spatial_network",
        number_of_simulations = as.integer(n_simulations)
      )
      if ("original" %in% names(enrich)) enrich <- enrich$original

      # Giotto versions can return either a matrix-like object or a long table.
      if (is.matrix(enrich)) return(enrich)
      enrich <- as.data.frame(enrich)
      from_col <- intersect(c("from", "cell_type", "celltype_1", "source"), names(enrich))[1]
      to_col <- intersect(c("to", "neighbor", "celltype_2", "target"), names(enrich))[1]
      score_col <- intersect(c("enrichm", "enrichment", "zscore", "PI_value", "score"), names(enrich))[1]
      if (any(is.na(c(from_col, to_col, score_col)))) {
        stop("Could not identify Giotto enrichment columns in cellProximityEnrichment output.")
      }
      labs <- sort(unique(c(as.character(enrich[[from_col]]), as.character(enrich[[to_col]]))))
      mat <- matrix(NA_real_, nrow = length(labs), ncol = length(labs), dimnames = list(labs, labs))
      for (i in seq_len(nrow(enrich))) {
        mat[as.character(enrich[[from_col]][i]), as.character(enrich[[to_col]][i])] <- as.numeric(enrich[[score_col]][i])
      }
      mat
    }
    """
    r_func = ro.r(r_code)
    r_input = df[["x", "y", "celltype"]].copy()
    r_input["celltype"] = r_input["celltype"].astype(str)
    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(r_input)
    r_matrix = r_func(r_df, int(k), int(n_simulations))
    values = np.asarray(r_matrix, dtype=float)
    row_names = list(r_matrix.rownames) if r_matrix.rownames is not None else sorted(r_input["celltype"].unique())
    col_names = list(r_matrix.colnames) if r_matrix.colnames is not None else row_names
    return pd.DataFrame(values, index=row_names, columns=col_names)


In [ ]:
# Panel mapping:
# - Supplementary Table 1, ANE benchmark helper. Wraps anhood/ANE so analytical neighborhood enrichment timing and memory can be recorded.
#

def compute_ane_matrix(df, anhood_repo=Path("/data/taobo.hu/projects/mouse_pup/analytical-enrichment-test")):
    """Run analytical neighborhood enrichment (ANE/anhood) on a point table."""
    import sys
    repo = Path(anhood_repo)
    if repo.exists() and str(repo) not in sys.path:
        sys.path.append(str(repo))
    from anhood.squidpy_wrapper import anhood

    adata = dataframe_to_adata(df)
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial")
    anhood(adata, cluster_key="celltype")
    cats = adata.obs["celltype"].cat.categories
    z = adata.uns["celltype_nhood_enrichment"]["zscore"]
    return pd.DataFrame(z, index=cats, columns=cats)


In [ ]:
# Panel mapping:
# - Supplementary Table 1, input-preparation cell. Loads mouse pup Xenium data and builds the common point table used by every benchmarked method.
#

import os
import time
import resource
import tracemalloc
import psutil
from sfplot import load_xenium_data, compute_cophenetic_distances_from_df

XENIUM_DIR = Path("Y:/long/10X_datasets/Xenium/Xenium_5K/Xenium_Prime_Mouse_Pup_FFPE_outs")
OUTPUT_DIR = Path("outputs/supp_table_1_runtime_memory")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
adata = load_xenium_data(str(XENIUM_DIR), normalize=False)
adata.obs["x"] = adata.obsm["spatial"][:, 0]
adata.obs["y"] = adata.obsm["spatial"][:, 1]
cluster_key = "annotation" if "annotation" in adata.obs else "Cluster"
points = adata.obs[["x", "y", cluster_key]].rename(columns={cluster_key: "celltype"})


In [ ]:
# Panel mapping:
# - Supplementary Table 1, measurement cell. Profiles CPU time, wall time, peak memory, page faults, and throughput for COSTE, Squidpy NE, Giotto, and ANE.
#

def benchmark(label, func, n_items):
    """Collect the runtime/memory fields reported in Supplementary Table 1."""
    process = psutil.Process(os.getpid())
    ru_start = resource.getrusage(resource.RUSAGE_SELF)
    rss_start = process.memory_info().rss
    cpu_start = time.process_time()
    wall_start = time.perf_counter()
    tracemalloc.start()
    func()
    _, peak_python = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    cpu_end = time.process_time()
    wall_end = time.perf_counter()
    rss_end = process.memory_info().rss
    ru_end = resource.getrusage(resource.RUSAGE_SELF)
    return {
        "method": label,
        "n_cells": n_items,
        "cpu_seconds": cpu_end - cpu_start,
        "wall_seconds": wall_end - wall_start,
        "peak_python_memory_gb": peak_python / 1024**3,
        "rss_delta_gb": (rss_end - rss_start) / 1024**3,
        "max_rss_gb": ru_end.ru_maxrss / 1024**2,
        "major_page_faults": ru_end.ru_majflt - ru_start.ru_majflt,
        "minor_page_faults": ru_end.ru_minflt - ru_start.ru_minflt,
        "throughput_cells_per_second": n_items / max(wall_end - wall_start, 1e-9),
    }

def run_coste():
    return compute_cophenetic_distances_from_df(points, x_col="x", y_col="y", celltype_col="celltype")


def run_squidpy_ne():
    tmp = dataframe_to_adata(points)
    sq.gr.spatial_neighbors(tmp, coord_type="generic", spatial_key="spatial")
    sq.gr.nhood_enrichment(tmp, cluster_key="celltype", n_perms=1000)
    return tmp.uns["celltype_nhood_enrichment"]["zscore"]


def run_giotto():
    return compute_giotto_matrix(points, n_simulations=1000, k=6)


def run_ane():
    return compute_ane_matrix(points)


rows = []
for label, func in [
    ("COSTE", run_coste),
    ("Squidpy NE", run_squidpy_ne),
    ("Giotto", run_giotto),
    ("ANE", run_ane),
]:
    try:
        rows.append(benchmark(label, func, adata.n_obs))
    except Exception as exc:
        rows.append({"method": label, "n_cells": adata.n_obs, "error": str(exc)})

pd.DataFrame(rows).to_csv(OUTPUT_DIR / "supplementary_table_1_runtime_memory.csv", index=False)
